In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U keras-hub keras
import os
os.environ["KERAS_BACKEND"] = "jax"   # the course uses JAX
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
import keras, keras_hub
print(keras.__version__, keras.backend.backend())

In [ ]:
model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
prompt = "<start_of_turn>user\nHow should I package dried mango for a small shop?<end_of_turn>\n<start_of_turn>model\n"
print(model.generate(prompt, max_length=200))

In [ ]:
import json

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

train, val, test = load("train.jsonl"), load("val.jsonl"), load("test.jsonl")
print(len(train), len(val), len(test))   # expect 166 19 15

def to_gemma(messages):
    # Gemma has no "system" role, so the system text goes at the start of the first user turn
    system, turns = messages[0]["content"], messages[1:]
    text = ""
    for i, m in enumerate(turns):
        if m["role"] == "user":
            content = (system + "\n\n" if i == 0 else "") + m["content"]
            text += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        else:
            text += f"<start_of_turn>model\n{m['content']}<end_of_turn>\n"
    return text

train_text = [to_gemma(r["messages"]) for r in train]
val_text   = [to_gemma(r["messages"]) for r in val]
test_text  = [to_gemma(r["messages"]) for r in test]

tok = model.preprocessor.tokenizer
lengths = [len(tok.tokenize(t)) for t in train_text]
print("longest example (tokens):", max(lengths), " average:", sum(lengths)//len(lengths))
print(train_text[0][:700])

In [ ]:
import numpy as np

model.preprocessor.sequence_length = 768   # longest example is 677 tokens
model.backbone.enable_lora(rank=8)         # freeze Gemma, add small trainable LoRA layers

trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
total = sum(int(np.prod(w.shape)) for w in model.weights)
print(f"trainable: {trainable:,}  total: {total:,}  share trained: {100*trainable/total:.2f}%")

In [ ]:
import keras, time

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)

t0 = time.time()
history = model.fit(train_text, validation_data=val_text,
                    epochs=3, batch_size=2, callbacks=[early])
print("training minutes:", round((time.time() - t0) / 60, 1))

In [ ]:
import tensorflow as tf

def to_pairs(messages):
    system, turns = messages[0]["content"], messages[1:]
    history, prompts, responses = "", [], []
    for i, m in enumerate(turns):
        if m["role"] == "user":
            content = (system + "\n\n" if i == 0 else "") + m["content"]
            history += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        else:
            prompts.append(history + "<start_of_turn>model\n")
            responses.append(m["content"] + "<end_of_turn>")
            history += f"<start_of_turn>model\n{m['content']}<end_of_turn>\n"
    return prompts, responses

def make_ds(rows, shuffle=False):
    P, R = [], []
    for r in rows:
        p, q = to_pairs(r["messages"]); P += p; R += q
    print(len(P), "prompt/response pairs")
    ds = tf.data.Dataset.from_tensor_slices({"prompts": P, "responses": R})
    if shuffle:
        ds = ds.shuffle(1000, seed=0)
    return ds.batch(2)

train_ds = make_ds(train, shuffle=True)
val_ds = make_ds(val)

t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=[early])
print("training minutes:", round((time.time() - t0) / 60, 1))

In [ ]:
!ls -la *.jsonl

In [ ]:
!pip install -q -U keras-hub keras
import os
os.environ["KERAS_BACKEND"] = "torch"
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")   # TensorFlow stays on the CPU; JAX keeps the GPU

import json, time, numpy as np, keras, keras_hub

model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b", dtype="float32")
model.preprocessor.sequence_length = 768
model.backbone.enable_lora(rank=8)

def load(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
train, val, test = load("train.jsonl"), load("val.jsonl"), load("test.jsonl")

def to_pairs(messages):
    system, turns = messages[0]["content"], messages[1:]
    history, prompts, responses = "", [], []
    for i, m in enumerate(turns):
        if m["role"] == "user":
            content = (system + "\n\n" if i == 0 else "") + m["content"]
            history += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        else:
            prompts.append(history + "<start_of_turn>model\n")
            responses.append(m["content"] + "<end_of_turn>")
            history += f"<start_of_turn>model\n{m['content']}<end_of_turn>\n"
    return prompts, responses

def make_ds(rows, shuffle=False):
    P, R = [], []
    for r in rows:
        p, q = to_pairs(r["messages"]); P += p; R += q
    print(len(P), "prompt/response pairs")
    ds = tf.data.Dataset.from_tensor_slices({"prompts": P, "responses": R})
    if shuffle: ds = ds.shuffle(1000, seed=0)
    return ds.batch(1)

train_ds, val_ds = make_ds(train, True), make_ds(val)
print("trainable:", sum(int(np.prod(w.shape)) for w in model.trainable_weights))

In [ ]:
import numpy as np
pre = model.preprocessor

def build(ds, name):
    xs, ys, sws = {}, [], []
    for i, batch in enumerate(ds):
        x, y, sw = model.preprocess_samples(batch)
        for k, v in x.items():
            if v is not None:
                xs.setdefault(k, []).append(keras.ops.convert_to_numpy(v))
        ys.append(keras.ops.convert_to_numpy(y))
        sws.append(keras.ops.convert_to_numpy(sw))
        if i % 100 == 0: print(name, i, flush=True)
    X = {k: np.concatenate(v) for k, v in xs.items()}
    return X, np.concatenate(ys), np.concatenate(sws).astype("float32")

Xtr, ytr, swtr = build(train_ds, "train")
Xva, yva, swva = build(val_ds, "val")
print("data ready:", {k: v.shape for k, v in Xtr.items()}, flush=True)

model.preprocessor = None   # data is already prepared, so skip the pipeline
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
print("compiled", flush=True)
sub = {k: v[:5] for k, v in Xtr.items()}
model.fit(x=sub, y=ytr[:5], sample_weight=swtr[:5], batch_size=1, epochs=1, verbose=2)
print("5 steps OK", flush=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.backbone.load_lora_weights("/content/drive/MyDrive/packaging_tutor/packaging_tutor_v3.lora.h5")
print("loaded")

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, "GB in use")

In [ ]:
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)

t0 = time.time()
history = model.fit(
    x=Xtr, y=ytr, sample_weight=swtr,
    validation_data=(Xva, yva, swva),
    batch_size=1, epochs=3, callbacks=[early], verbose=2,
)
print("training minutes:", round((time.time() - t0) / 60, 1))

In [ ]:
h = history.history
print("loss    :", [round(x, 3) for x in h["loss"]])
print("val_loss:", [round(x, 3) for x in h["val_loss"]])

In [ ]:
import json
from google.colab import files

model.backbone.save_lora_weights("packaging_tutor_1b.lora.h5")   # the trained LoRA layers only
json.dump(history.history, open("history_1b.json", "w"))
files.download("packaging_tutor_1b.lora.h5")
files.download("history_1b.json")

In [ ]:
model.preprocessor = pre   # put the text preparation back for chatting
system = train[0]["messages"][0]["content"]

def chat_prompt(turns):
    text = ""
    for i, (role, content) in enumerate(turns):
        if role == "user":
            c = (system + "\n\n" if i == 0 else "") + content
            text += f"<start_of_turn>user\n{c}<end_of_turn>\n"
        else:
            text += f"<start_of_turn>model\n{content}<end_of_turn>\n"
    return text + "<start_of_turn>model\n"

p = chat_prompt([("user", "How should I package dried mango for a small shop?")])
print(model.generate(p, max_length=600)[len(p):])

In [ ]:
import json, csv, time
model.preprocessor = pre

test = [json.loads(l) for l in open("test.jsonl", encoding="utf-8")]
P, R, ids = [], [], []
for r in test:
    p, q = to_pairs(r["messages"])
    P += p; R += q; ids += [r["id"]] * len(p)

step = max(1, len(P) // 40)          # about 40 samples, spread across the test set
rows, t0 = [], time.time()
for i in range(0, len(P), step):
    out = model.generate(P[i], max_length=768)
    reply = out[len(P[i]):].split("<end_of_turn>")[0].strip()
    rows.append([ids[i], P[i][-300:], R[i], reply])
    if len(rows) % 10 == 0: print(len(rows), "done")

with open("/content/drive/MyDrive/packaging_tutor/eval_v3.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id", "prompt_end", "reference", "model_reply"]); w.writerows(rows)
print(len(rows), "replies saved,", round((time.time() - t0) / 60, 1), "min")

In [ ]:
import textwrap

def ask(q):
    p = chat_prompt([("user", q)])
    out = model.generate(p, max_length=600)[len(p):]
    print("Q:", q); print(textwrap.fill(out, 100)); print("-" * 100)

ask("How should I package dried mango for a small shop?")
ask("I sell sachet water to street sellers and want to reduce plastic waste. Where do I start?")
ask("Can you write my history essay?")

In [ ]:
PACK = ["packag","bottle","jar","sachet","pouch","carton","box","crate","sack","bag","label","shelf",
        "material","plastic","glass","pet ","hdpe","ldpe","polythene","recycl","waste","cost","fda","standard",
        "product","tub","lid","seal","film","container","tin ","juice","shea","gari","palm oil","shito","tomato",
        "honey","mango","soap","water","kenkey","groundnut","fish","egg","bread","yoghurt","sobolo","soy",
        "coconut","chocolate","pepper","cashew","herbal","fufu","pineapple","plantain","pomade","cereal"]
OFF = ["essay","poem","homework","medicine","malaria","python","code ","translate","politic","vote","exam",
       "cheat","recipe","weather","story","girlfriend","hack","cover letter","cv ","workout","crypto","stock",
       "write my","do my","assignment","ignore your","system instructions"]
DECLINE = ("That's outside what I can help with here. I'm a packaging teaching assistant, so I can guide you "
           "on materials, costs, regulations to check and sustainability. What product are you working on?")

def guarded_ask(turns):
    last = turns[-1][1].lower() + " "
    first = len(turns) == 1
    if any(w in last for w in OFF) or (first and not any(w in last for w in PACK)):
        return DECLINE
    p = chat_prompt(turns)
    return model.generate(p, max_length=600)[len(p):]

for q in ["Can you write my history essay?",
          "How should I package dried mango for a small shop?",
          "What medicine should I take for malaria?",
          "I sell sachet water to street sellers and want to reduce plastic waste. Where do I start?"]:
    print("Q:", q); print(textwrap.fill(guarded_ask([("user", q)]), 100)); print("-" * 100)

In [ ]:
model.compile(sampler="greedy")   # always pick the most likely next word

for q in ["How should I package dried mango for a small shop?",
          "I sell sachet water to street sellers and want to reduce plastic waste. Where do I start?",
          "I make shea butter in a cooperative and sell in Accra. What pack should I use?"]:
    print("Q:", q); print(textwrap.fill(guarded_ask([("user", q)]), 100)); print("-" * 100)

mt = [("user", "How should I package dried mango for a small shop?"),
      ("model", "Good place to start. Before we compare materials, I need two things. How will you dry and store the mango, and where will customers buy it? Once I know that, we'll compare options together."),
      ("user", "It is dried and kept in a cool room, and I sell to shops in my town. I think a plain paper bag is best.")]
print("MULTI-TURN"); print(textwrap.fill(guarded_ask(mt), 100))

In [ ]:
REG = ["regulat","fda","standard","label","legal","safe","complian","approved","certif","expiry","permit"]
NOTE = ("\n\nReminder: I can't certify compliance. Please verify any regulatory or safety point with "
        "Ghana FDA, the Ghana Standards Authority and your instructor.")

def final_reply(turns):
    reply = guarded_ask(turns)
    if reply == DECLINE: return reply
    text = (turns[-1][1] + " " + reply).lower()
    return reply + NOTE if any(w in text for w in REG) else reply

print(textwrap.fill(final_reply([("user", "Is my juice label legal and approved?")]), 100))

In [ ]:
VERDICT = ["legal", "approved", "complian", "certif", "allowed", "permitted", "registered", "registration",
           "is it safe", "is this safe", "safe for", "safe to", "meets the standard", "meet the standard", "pass inspection"]
GATE = ("I can't tell you whether something is legal, approved or safe, and I don't want you to rely on me for that. "
        "Please ask Ghana FDA and the Ghana Standards Authority, and check with your instructor. "
        "To prepare, write down what your product, pack and label say: the product name, ingredients, quantity, "
        "who made it, dates and storage. Which of those do you not have yet?")

def final_reply(turns):
    last = turns[-1][1].lower()
    if any(w in last for w in VERDICT):
        return GATE
    reply = guarded_ask(turns)
    if reply == DECLINE: return reply
    text = (last + " " + reply).lower()
    return reply + NOTE if any(w in text for w in REG) else reply

for q in ["Is my juice label legal and approved?",
          "Does my honey jar need registration before I sell it?",
          "Is PET safe for hot juice?",
          "How should I package dried mango for a small shop?",
          "Can you write my history essay?"]:
    print("Q:", q); print(textwrap.fill(final_reply([("user", q)]), 100)); print("-" * 100)

In [ ]:
print(final_reply([("user", "Write my history essay.")]))

In [ ]:
t = [("user", "I sell shea butter in Accra. What pack should I use?")]
r = final_reply(t); print(r, "\n")
t += [("model", r), ("user", "It's for a small cosmetics shop in Kumasi, hot weather, about 100 tubs a week.")]
print(final_reply(t))

In [ ]:
def guard_blocks(turns):
    last = turns[-1][1].lower() + " "
    return any(w in last for w in OFF) or (len(turns) == 1 and not any(w in last for w in PACK))

In [ ]:
print(OFF)

In [ ]:
import re

# words that can appear in normal packaging talk: they only block when the message has no packaging word
SOFT = {"weather", "recipe", "medicine", "exam", "code"}

def guard_blocks(turns):
    last = turns[-1][1].lower() + " "
    on_topic = any(w in last for w in PACK)
    hits = [w.strip() for w in OFF if re.search(r"\b" + re.escape(w.strip()), last)]
    if any(w not in SOFT for w in hits):     # clearly off-topic words always block
        return True
    if len(turns) == 1 and not on_topic:     # first message must mention packaging
        return True
    return False

In [ ]:
print(final_reply([("user", "Write my history essay.")]), "\n")
print(final_reply([("user", "What's the weather in Accra today?")]), "\n")
t = [("user", "I sell shea butter in Accra. What pack should I use?")]
r = final_reply(t); print(r, "\n")
t += [("model", r), ("user", "It's for a small cosmetics shop in Kumasi, hot weather, about 100 tubs a week.")]
print(final_reply(t))

In [ ]:
import re, textwrap
msg = "It's for a small cosmetics shop in Kumasi, hot weather, about 100 tubs a week.".lower() + " "
print("matching OFF words:", [w for w in OFF if re.search(r"\b" + re.escape(w.strip()), msg)])
print()
print(textwrap.fill(str(OFF), 100))

In [ ]:
t2 = [("user", "I sell shea butter in Accra. What pack should I use?"),
      ("model", "Let's take it step by step."),
      ("user", "It's for a small cosmetics shop in Kumasi, hot weather, about 100 tubs a week.")]
print("guard_blocks says:", guard_blocks(t2))
print("raw model reply:")
print(clean(model.generate(chat_prompt(t2), max_length=768)[len(chat_prompt(t2)):]))

In [ ]:
import inspect
print(inspect.getsource(guarded_ask))
print("-----")
print(inspect.getsource(final_reply))

In [ ]:
def guarded_ask(turns):
    if guard_blocks(turns):
        return DECLINE
    p = chat_prompt(turns)
    return model.generate(p, max_length=600)[len(p):]

In [ ]:
print(final_reply([("user", "Write my history essay.")]))

In [ ]:
!pip -q install gradio
import gradio as gr, csv, os, datetime

LOG = "feedback.csv"
if not os.path.exists(LOG):
    with open(LOG, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["time", "student_message", "tutor_reply",
                                "rating_1_to_5", "issue_type", "comment"])

def render(turns):
    return "\n\n".join(("YOU: " if r == "user" else "TUTOR: ") + t for r, t in turns)

def send(msg, turns):
    msg = (msg or "").strip()
    if not msg:
        return turns, render(turns), "", ""
    turns = turns + [("user", msg)]
    reply = final_reply(turns).replace("<end_of_turn>", "").strip()
    turns = turns + [("model", reply)]
    return turns, render(turns), "", reply

def save_feedback(turns, rating, issue, comment):
    if len(turns) < 2:
        return "Send a message first, then rate the reply."
    with open(LOG, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([datetime.datetime.now().isoformat(timespec="seconds"),
                                turns[-2][1], turns[-1][1], rating, issue, comment])
    return "Thank you, feedback saved."

def reset():
    return [], "", "", ""

with gr.Blocks(title="Packaging Tutor (prototype)") as demo:
    gr.Markdown("# Packaging Technology Teaching Assistant (prototype)\n"
                "A student-project model (Gemma 3 270M + LoRA). **It can be wrong.** "
                "Do not rely on it for regulations or safety; verify with Ghana FDA / "
                "Ghana Standards Authority and your instructor. "
                "Please do not type personal information.")
    turns = gr.State([])
    box = gr.Textbox(label="Conversation", lines=14, interactive=False)
    inp = gr.Textbox(label="Your message", placeholder="e.g. I sell shea butter in Accra. What pack should I use?")
    last = gr.Textbox(visible=False)
    with gr.Row():
        b_send = gr.Button("Send", variant="primary")
        b_new = gr.Button("New conversation")
    gr.Markdown("### Feedback on the last reply")
    rating = gr.Radio(["1", "2", "3", "4", "5"], label="Rating (5 = very good)")
    issue = gr.Dropdown(["None, it was good", "Wrong or unclear facts", "Did not ask questions / just gave answers",
                         "Lost the thread of the conversation", "Should have declined but answered",
                         "Declined when it should have helped", "Too long or too short", "Other"],
                        label="Main issue")
    comment = gr.Textbox(label="What should we improve?", lines=2)
    b_fb = gr.Button("Submit feedback")
    status = gr.Markdown()

    b_send.click(send, [inp, turns], [turns, box, inp, last])
    inp.submit(send, [inp, turns], [turns, box, inp, last])
    b_new.click(reset, None, [turns, box, inp, last])
    b_fb.click(save_feedback, [turns, rating, issue, comment], status)

demo.queue().launch(share=True, debug=False)

In [ ]:
from google.colab import files
files.download("feedback.csv")

In [ ]:
NEG = ["not looking for", "don't want", "do not want", "not asking", "no need for", "not interested in"]

def final_reply(turns):
    last = turns[-1][1].lower()
    if any(w in last for w in VERDICT) and not any(n in last for n in NEG):
        return GATE
    reply = guarded_ask(turns)
    if reply == DECLINE:
        return reply
    text = (last + " " + reply).lower()
    already = any(p in reply.lower() for p in ["please verify", "can't certify"])
    return reply + NOTE if any(w in text for w in REG) and not already else reply

In [ ]:
import os
print("working folder:", os.getcwd())
print(os.listdir("."))
!find / -name "feedback*.csv" 2>/dev/null